# 6B. Per-Exercise Sequence-Length Sweep

This notebook starts from the frozen shared Stage 6 baseline and tests whether sequence length is the main bottleneck for the most promising exercises.


## Reason, Approach, Result Interpretation

**Reason**
- The first shared Stage 6 baseline improved on a trivial train-mean baseline for several exercises, but absolute errors remained high.
- Before changing representations or adding exercise-specific keypoint weighting, we should test whether temporal compression is the main bottleneck.

**Approach**
- Keep the same pose-sequence data contract and TCN family.
- Sweep a small set of `seq_len` values for the most promising exercises.
- Compare both raw validation metrics and baseline-comparison deltas.

**Result interpretation**
- If longer sequence lengths improve both `MAE` and `Within-1`, temporal compression is likely a real bottleneck.
- If sequence length changes little, the next likely lever is exercise-specific keypoint emphasis or a different representation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Environment Setup

**Why this section exists**
- The sweep needs the current trainer, baseline-comparison script, and the full Stage 5 sequence dataset in the same Colab session.

**Approach**
- Mount Drive.
- Sync the current repo copies of the Stage 6 trainer and comparison script into the Drive project.
- Resolve the shared sequence index that will be reused for every sequence-length run.

**How to interpret the result**
- If the paths print correctly and `SEQUENCE_INDEX exists = True`, the environment is ready.
- If the index is missing, rerun Stage 5 before starting the sweep.


In [ ]:
from pathlib import Path
import shutil

CODE_ROOT = Path('/content/CV_Image_pose_detection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')

TRAINER_REL = Path('artifacts/3_Modeling/train_pose_count_tcn.py')
COMPARE_REL = Path('artifacts/3_Modeling/compare_count_run_to_baseline.py')
TRAINER_SRC = CODE_ROOT / TRAINER_REL
TRAINER_DST = DRIVE_PROJECT_ROOT / TRAINER_REL
COMPARE_SRC = CODE_ROOT / COMPARE_REL
COMPARE_DST = DRIVE_PROJECT_ROOT / COMPARE_REL

if TRAINER_SRC.exists():
    TRAINER_DST.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(TRAINER_SRC, TRAINER_DST)

if COMPARE_SRC.exists():
    COMPARE_DST.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(COMPARE_SRC, COMPARE_DST)

ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/annotation_cleaned'
SEQUENCE_INDEX = ANNOTATION_DIR / 'pose_sequence_index.csv'

print('TRAINER_DST =', TRAINER_DST)
print('COMPARE_DST =', COMPARE_DST)
print('SEQUENCE_INDEX exists =', SEQUENCE_INDEX.exists())


## Dataset Coverage Check

**Why this section exists**
- The sweep should only run after the full pose-sequence dataset has been rebuilt, including squat.
- A missing `train` or `valid` split would make the exercise unusable for this experiment.

**Approach**
- Read `pose_sequence_index.csv`.
- Count how many `train`, `valid`, and `test` rows exist for each exercise.

**How to interpret the result**
- Each target exercise should have nonzero `train` and `valid` rows.
- If an exercise is missing, the issue is still upstream in Stage 5 rather than in the TCN sweep itself.


In [ ]:
import pandas as pd

seq_df = pd.read_csv(SEQUENCE_INDEX)
counts_df = seq_df.groupby(['type', 'split']).size().unstack(fill_value=0).sort_index()
display(counts_df)


## Sweep Design

**Why this section exists**
- We want to test whether temporal compression is the main bottleneck before changing representations or keypoint weighting.

**Approach**
- Keep the model family and training recipe fixed.
- Change only `seq_len` across a small set of values.
- Focus on the exercises that looked most informative or most important in the frozen shared baseline.

**How to interpret the result**
- If one `seq_len` consistently improves `MAE` and `Within-1`, that suggests temporal resolution matters.
- If all `seq_len` values behave similarly, the next lever is more likely exercise-specific feature emphasis or a different representation.


In [ ]:
EXERCISES = ['pull_up', 'bench_pressing', 'pommelhorse', 'push_up', 'squat']
SEQ_LENS = [128, 192, 256, 384]

EPOCHS = 80
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-4
CHANNELS = 96
KERNEL_SIZE = 3
NUM_BLOCKS = 4
DROPOUT = 0.2
PATIENCE = 15
LOSS = 'l1'
EVAL_TRANSFORM = 'raw'
SELECTION_METRIC = 'mae'
SAMPLER = 'balanced_count'
TIME_WARP_RANGE = 0.12
FEATURE_NOISE_STD = 0.02
FRAME_DROPOUT_PROB = 0.03

RUNS = []
for exercise in EXERCISES:
    for seq_len in SEQ_LENS:
        RUNS.append({
            'exercise': exercise,
            'seq_len': seq_len,
            'run_name': f'pose_count_tcn_{exercise}_seq{seq_len}',
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'lr': LR,
            'weight_decay': WEIGHT_DECAY,
            'channels': CHANNELS,
            'kernel_size': KERNEL_SIZE,
            'num_blocks': NUM_BLOCKS,
            'dropout': DROPOUT,
            'patience': PATIENCE,
            'loss': LOSS,
            'eval_transform': EVAL_TRANSFORM,
            'selection_metric': SELECTION_METRIC,
            'sampler': SAMPLER,
            'time_warp_range': TIME_WARP_RANGE,
            'feature_noise_std': FEATURE_NOISE_STD,
            'frame_dropout_prob': FRAME_DROPOUT_PROB,
        })

pd.DataFrame(RUNS)


## Training Execution

**Why this section exists**
- This is the actual sweep stage: it launches one run per exercise and sequence length.

**Approach**
- Reuse the same trainer and configuration family from the frozen shared baseline.
- Keep failures non-fatal so one bad run does not stop the full grid.

**How to interpret the result**
- A clean sweep means every configured run finished and wrote artifacts.
- If some rows fail, they should be treated as isolated debugging targets, not as evidence that the whole sweep failed.


In [ ]:
import subprocess
import pandas as pd

training_failures = []
for cfg in RUNS:
    cmd = [
        'python', str(TRAINER_DST),
        '--project-dir', str(DRIVE_PROJECT_ROOT),
        '--index-csv', str(SEQUENCE_INDEX),
        '--run-name', cfg['run_name'],
        '--exercise', cfg['exercise'],
        '--seq-len', str(cfg['seq_len']),
        '--epochs', str(cfg['epochs']),
        '--batch-size', str(cfg['batch_size']),
        '--lr', str(cfg['lr']),
        '--weight-decay', str(cfg['weight_decay']),
        '--channels', str(cfg['channels']),
        '--kernel-size', str(cfg['kernel_size']),
        '--num-blocks', str(cfg['num_blocks']),
        '--dropout', str(cfg['dropout']),
        '--patience', str(cfg['patience']),
        '--loss', cfg['loss'],
        '--eval-transform', cfg['eval_transform'],
        '--selection-metric', cfg['selection_metric'],
        '--sampler', cfg['sampler'],
        '--time-warp-range', str(cfg['time_warp_range']),
        '--feature-noise-std', str(cfg['feature_noise_std']),
        '--frame-dropout-prob', str(cfg['frame_dropout_prob']),
        '--device', 'cuda',
    ]
    print('\nRunning:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        training_failures.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'run_name': cfg['run_name'],
            'returncode': exc.returncode,
        })
        print(f"FAILED: {cfg['exercise']} seq_len={cfg['seq_len']} (returncode={exc.returncode})")

if training_failures:
    display(pd.DataFrame(training_failures))
else:
    print('All runs completed successfully.')


## Raw Metric Review

**Why this section exists**
- The first pass should examine the direct validation metrics before comparing against any trivial baseline.

**Approach**
- Load each run's `metrics_summary.json`.
- Compare `valid_mae`, `valid_rmse`, and `valid_within_1` across sequence lengths for each exercise.

**How to interpret the result**
- The best row per exercise gives the strongest raw sequence-length setting under the current model family.
- This section answers whether sequence length alone can improve the absolute counting error.


In [ ]:
import json
import pandas as pd

rows = []
for cfg in RUNS:
    metrics_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / cfg['run_name'] / 'metrics_summary.json'
    if not metrics_path.exists():
        continue
    with open(metrics_path, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    rows.append({
        'exercise': cfg['exercise'],
        'seq_len': cfg['seq_len'],
        'run_name': cfg['run_name'],
        'best_epoch': metrics.get('best_epoch'),
        'valid_mae': metrics['valid_metrics']['mae'],
        'valid_rmse': metrics['valid_metrics']['rmse'],
        'valid_within_1': metrics['valid_metrics']['within_1'],
    })

results_df = pd.DataFrame(rows)
if results_df.empty:
    print('No metrics_summary.json files found yet.')
else:
    display(results_df.sort_values(['exercise', 'seq_len']))
    best_by_exercise = results_df.sort_values(['exercise', 'valid_mae']).groupby('exercise', as_index=False).first()
    display(best_by_exercise)


## Baseline-Comparison Review

**Why this section exists**
- A lower raw `MAE` is useful, but the more important question is whether the model is truly adding value over a trivial counting baseline.

**Approach**
- Compare each run to the train-split mean-count baseline using the same valid rows.
- Review both `delta_mae` and `delta_within_1`.

**How to interpret the result**
- Negative `delta_mae` means the TCN beat the trivial baseline on average error.
- Positive `delta_within_1` means the TCN landed within one repetition more often than the baseline.
- The best next configuration is the one that improves both the raw metrics and the baseline comparison in a consistent way.


In [ ]:
import subprocess
import json
import pandas as pd

baseline_rows = []
comparison_failures = []
for cfg in RUNS:
    run_dir = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / cfg['run_name']
    pred_path = run_dir / 'predictions.csv'
    if not pred_path.exists():
        continue
    cmd = [
        'python', str(COMPARE_DST),
        '--index-csv', str(SEQUENCE_INDEX),
        '--predictions-csv', str(pred_path),
        '--exercise', cfg['exercise'],
    ]
    print('\nComparing:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        comparison_failures.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'run_name': cfg['run_name'],
            'returncode': exc.returncode,
        })
        continue
    summary_path = run_dir / 'baseline_comparison_summary.json'
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    baseline_rows.append({
        'exercise': cfg['exercise'],
        'seq_len': cfg['seq_len'],
        'run_name': cfg['run_name'],
        'model_mae': summary['model_metrics']['mae'],
        'baseline_mae': summary['baseline_metrics']['mae'],
        'delta_mae': summary['delta_vs_baseline']['mae'],
        'model_within_1': summary['model_metrics']['within_1'],
        'baseline_within_1': summary['baseline_metrics']['within_1'],
        'delta_within_1': summary['delta_vs_baseline']['within_1'],
        'model_beats_baseline': summary['row_level']['model_beats_baseline'],
        'valid_rows': summary['row_level']['valid_rows'],
    })

baseline_df = pd.DataFrame(baseline_rows)
if baseline_df.empty:
    print('No baseline comparison summaries found yet.')
else:
    display(baseline_df.sort_values(['exercise', 'seq_len']))
    best_delta = baseline_df.sort_values(['exercise', 'delta_mae']).groupby('exercise', as_index=False).first()
    display(best_delta)

if comparison_failures:
    display(pd.DataFrame(comparison_failures))
